## 🗺️ Parte Adicional: Análisis Geoespacial con H3

En esta sección aprenderemos a:
- Agregar datos por zonas geográficas (hexágonos H3)
- Filtrar clientes por proximidad a sucursales
- Realizar joins espaciales entre ventas y ubicaciones
- Analizar patrones geográficos en el comportamiento de compra

In [0]:
import h3

# Unir ventas con clientes para obtener ubicación
ventas_con_ubicacion = df_ventas[df_ventas['cliente_id'].notna()].merge(
    df_clientes[['cliente_id', 'latitud', 'longitud', 'h3_index']], 
    on='cliente_id',
    how='left'
)

print("📍 VENTAS CON INFORMACIÓN GEOESPACIAL")
print("=" * 60)
print(f"Total de ventas: {len(df_ventas):,}")
print(f"Ventas con ubicación conocida: {len(ventas_con_ubicacion):,} ({len(ventas_con_ubicacion)/len(df_ventas)*100:.1f}%)")
print(f"\nPrimeras 5 ventas con ubicación:")
ventas_con_ubicacion[['venta_id', 'fecha', 'total', 'cliente_id', 'latitud', 'longitud', 'h3_index']].head()

In [0]:
# Agrupar ventas por zona geográfica (h3_index)
ventas_por_zona = ventas_con_ubicacion.groupby('h3_index').agg({
    'venta_id': 'count',
    'total': ['sum', 'mean']
}).round(2)

ventas_por_zona.columns = ['Num_Ventas', 'Facturacion_Total', 'Ticket_Promedio']
ventas_por_zona = ventas_por_zona.sort_values('Facturacion_Total', ascending=False)

print("📊 FACTURACIÓN POR ZONA GEOGRÁFICA (Top 10 Hexágonos H3)")
print("=" * 70)
print(ventas_por_zona.head(10))

print("\n💡 INTERPRETACIÓN:")
print(f"   - Zona con mayor facturación: ${ventas_por_zona['Facturacion_Total'].max():,.2f}")
print(f"   - Zona con menor facturación: ${ventas_por_zona['Facturacion_Total'].min():,.2f}")
print(f"   - Zonas únicas con ventas: {len(ventas_por_zona)}")

In [0]:
# Calcular densidad de clientes por zona
clientes_por_zona = df_clientes.groupby('h3_index').size().reset_index(name='num_clientes')
clientes_por_zona = clientes_por_zona.sort_values('num_clientes', ascending=False)

print("👥 DENSIDAD DE CLIENTES POR ZONA (Top 10)")
print("=" * 60)
for idx, row in clientes_por_zona.head(10).iterrows():
    print(f"Zona {row['h3_index']}: {row['num_clientes']} clientes")

# Estadisticas de distribución
print("\n📊 ESTADÍSTICAS DE DISTRIBUCIÓN:")
print(f"   - Zonas totales: {len(clientes_por_zona)}")
print(f"   - Promedio de clientes por zona: {clientes_por_zona['num_clientes'].mean():.2f}")
print(f"   - Desviación estándar: {clientes_por_zona['num_clientes'].std():.2f}")
print(f"   - Zona más densa: {clientes_por_zona['num_clientes'].max()} clientes")
print(f"   - Zona menos densa: {clientes_por_zona['num_clientes'].min()} cliente(s)")

In [0]:
# Ejemplo: Encontrar clientes cercanos a la sucursal Centro
sucursal_centro = df_sucursales[df_sucursales['nombre'] == 'Espiga Dorada - Centro'].iloc[0]
print(f"🏪 Analizando: {sucursal_centro['nombre']}")
print(f"   H3 Index: {sucursal_centro['h3_index']}")

# Obtener hexágonos vecinos (radio 3 = ~300m)
vecinos = h3.grid_disk(sucursal_centro['h3_index'], 3)
print(f"\n🔶 Área de búsqueda: {len(vecinos)} hexágonos (radio 3)")

# Filtrar clientes en esa área
clientes_cercanos = df_clientes[df_clientes['h3_index'].isin(vecinos)]

print(f"\n👥 Clientes cercanos encontrados: {len(clientes_cercanos)}")
print(f"   Representa el {len(clientes_cercanos)/len(df_clientes)*100:.1f}% del total")
print(f"\nPrimeros 5 clientes cercanos:")
print(clientes_cercanos[['cliente_id', 'nombre', 'h3_index']].head())

In [0]:
# Para cada sucursal, calcular ventas en su área de influencia
print("📊 VENTAS POR ÁREA DE INFLUENCIA DE SUCURSAL")
print("=" * 70)

for idx, sucursal in df_sucursales.iterrows():
    # Área de influencia (radio 5 hexágonos)
    area_influencia = h3.grid_disk(sucursal['h3_index'], 5)
    
    # Clientes en esa área
    clientes_area = df_clientes[df_clientes['h3_index'].isin(area_influencia)]
    
    # Ventas de esos clientes
    ventas_area = ventas_con_ubicacion[
        ventas_con_ubicacion['cliente_id'].isin(clientes_area['cliente_id'])
    ]
    
    print(f"\n{sucursal['nombre']} ({sucursal['zona']})")
    print(f"   👥 Clientes en área: {len(clientes_area)}")
    print(f"   📋 Ventas registradas: {len(ventas_area):,}")
    print(f"   💰 Facturación total: ${ventas_area['total'].sum():,.2f}")
    print(f"   🎯 Ticket promedio: ${ventas_area['total'].mean():,.2f}")

print("\n💡 INSIGHT: Comparar facturación por área vs. ubicación física de sucursal")

### ✅ Ejercicio Práctico: Análisis de Zonas Sin Cobertura

**Objetivo:** Identificar zonas con clientes pero lejos de todas las sucursales.

**Pasos:**
1. Para cada cliente, calcular la distancia mínima a cualquier sucursal
2. Identificar clientes a más de 10 hexágonos de distancia
3. Agrupar por zona H3 y calcular potencial de negocio

**Usa:** `h3.grid_distance(h3_a, h3_b)` para calcular distancias

In [0]:
# Para cada cliente, calcular distancia mínima a sucursales
print("🔍 IDENTIFICANDO ZONAS SIN COBERTURA ADECUADA")
print("=" * 70)

distancias_minimas = []

for idx, cliente in df_clientes.iterrows():
    distancias_a_sucursales = [
        h3.grid_distance(cliente['h3_index'], suc['h3_index'])
        for _, suc in df_sucursales.iterrows()
    ]
    dist_min = min(distancias_a_sucursales)
    distancias_minimas.append({
        'cliente_id': cliente['cliente_id'],
        'h3_index': cliente['h3_index'],
        'distancia_min': dist_min
    })

df_distancias = pd.DataFrame(distancias_minimas)

# Clientes lejos de sucursales (>10 hexágonos = ~1km)
clientes_lejos = df_distancias[df_distancias['distancia_min'] > 10]

print(f"\n🚨 Clientes lejos de sucursales (>10 hexágonos): {len(clientes_lejos)} ({len(clientes_lejos)/len(df_clientes)*100:.1f}%)")

# Agrupar por zona para identificar oportunidades
if len(clientes_lejos) > 0:
    zonas_desatendidas = clientes_lejos.groupby('h3_index').agg({
        'cliente_id': 'count',
        'distancia_min': 'mean'
    }).sort_values('cliente_id', ascending=False)
    zonas_desatendidas.columns = ['Num_Clientes', 'Distancia_Promedio']
    
    print("\n📍 Top 5 zonas con mayor potencial (clientes lejos):")
    print(zonas_desatendidas.head())
    
    print("\n💡 RECOMENDACIÓN: Considerar abrir una nueva sucursal en estas zonas")
else:
    print("\n✅ Todos los clientes están relativamente cerca de alguna sucursal")

# TP02: Manipulación Programática y Exploración
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 1: Herramientas en la Nube de Análisis de Datos

---

### 🎯 Objetivos del Trabajo Práctico

En este trabajo práctico aprenderemos a:

1. **Cargar datasets** desde archivos CSV usando Pandas
2. **Transformar estructuras de datos** con operaciones de Pandas
3. **Limpiar y preparar datos** para análisis
4. **Explorar información** integrando celdas SQL
5. **Realizar análisis exploratorio** básico (EDA)

---

### 📁 Caso de Estudio: Análisis de Ventas de Panadería

Continuamos trabajando con los datos de la **Panadería La Espiga Dorada**. En este TP nos enfocaremos en:

* Cargar y unir múltiples datasets (ventas, productos, clientes)
* Limpiar datos inconsistentes
* Crear nuevas variables derivadas
* Explorar patrones de ventas

---

### 🕰️ Duración Estimada: 2 horas

## Parte 1: Carga de Datos con Pandas

### 📂 Lectura de múltiples archivos CSV

Vamos a cargar todos los datasets de la panadería usando Pandas.

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

# Ruta base de los datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

# Cargar todos los datasets
print("📂 Cargando datasets...\n")

df_productos = pd.read_csv(ruta_datos + 'productos.csv')
print(f"✅ Productos: {len(df_productos)} registros")

df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
print(f"✅ Sucursales: {len(df_sucursales)} registros")

df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')
print(f"✅ Clientes: {len(df_clientes)} registros")

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
print(f"✅ Ventas: {len(df_ventas)} registros")

df_detalles_ventas = pd.read_csv(ruta_datos + 'detalles_ventas.csv')
print(f"✅ Detalles de ventas: {len(df_detalles_ventas)} registros")

print("\n✅ Todos los datasets cargados exitosamente")

In [0]:
# Convertir columnas de fecha a datetime
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_clientes['fecha_registro'] = pd.to_datetime(df_clientes['fecha_registro'])
df_sucursales['fecha_apertura'] = pd.to_datetime(df_sucursales['fecha_apertura'])

print("✅ Columnas de fecha convertidas a datetime")
print("\n📊 Tipos de datos actualizados:")
print(f"  ventas.fecha: {df_ventas['fecha'].dtype}")
print(f"  clientes.fecha_registro: {df_clientes['fecha_registro'].dtype}")
print(f"  sucursales.fecha_apertura: {df_sucursales['fecha_apertura'].dtype}")

## Parte 2: Transformaciones con Pandas

### 🔄 Operaciones de transformación

Vamos a crear nuevas columnas y transformar los datos existentes para hacerlos más útiles para el análisis.

In [0]:
# Crear columnas derivadas en el dataset de ventas
df_ventas['anio'] = df_ventas['fecha'].dt.year
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['dia'] = df_ventas['fecha'].dt.day
df_ventas['dia_semana'] = df_ventas['fecha'].dt.day_name()
df_ventas['numero_dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['es_fin_de_semana'] = df_ventas['numero_dia_semana'] >= 5
df_ventas['trimestre'] = df_ventas['fecha'].dt.quarter

print("✅ Columnas temporales creadas en df_ventas")
print("\n📅 Nuevas columnas:")
print(df_ventas[['fecha', 'anio', 'mes', 'dia_semana', 'es_fin_de_semana', 'trimestre']].head(10))

In [0]:
# Crear categorías de precio en productos
def categorizar_precio(precio):
    if precio < 500:
        return 'Económico'
    elif precio < 1500:
        return 'Medio'
    elif precio < 5000:
        return 'Premium'
    else:
        return 'Lujo'

df_productos['rango_precio'] = df_productos['precio_unitario'].apply(categorizar_precio)

print("✅ Categorías de precio creadas")
print("\n💰 Distribución por rango de precio:")
print(df_productos['rango_precio'].value_counts().sort_index())

## Parte 3: Limpieza de Datos

### 🧽 Manejo de valores faltantes

Vamos a identificar y manejar los valores faltantes en nuestros datasets.

In [0]:
# Analizar valores faltantes en clientes
print("🔍 VALORES FALTANTES EN CLIENTES")
print("=" * 60)

nulos_clientes = df_clientes.isnull().sum()
porcentaje_nulos = (nulos_clientes / len(df_clientes) * 100).round(2)

resumen_nulos = pd.DataFrame({
    'Columna': nulos_clientes.index,
    'Valores_Faltantes': nulos_clientes.values,
    'Porcentaje': porcentaje_nulos.values
})

print(resumen_nulos[resumen_nulos['Valores_Faltantes'] > 0])

print(f"\n📊 Total de clientes: {len(df_clientes)}")
print(f"Clientes sin email: {df_clientes['email'].isnull().sum()}")
print(f"Clientes sin teléfono: {df_clientes['telefono'].isnull().sum()}")
print(f"Clientes sin preferencia: {df_clientes['preferencia_categoria'].isnull().sum()}")

In [0]:
# Crear una copia del dataframe para trabajar
df_clientes_limpio = df_clientes.copy()

# Llenar preferencia_categoria con 'Sin Preferencia'
df_clientes_limpio['preferencia_categoria'] = df_clientes_limpio['preferencia_categoria'].fillna('Sin Preferencia')

# Marcar clientes con datos de contacto completos
df_clientes_limpio['tiene_contacto_completo'] = (
    df_clientes_limpio['email'].notna() & 
    df_clientes_limpio['telefono'].notna()
)

print("✅ Valores faltantes manejados")
print("\n📊 Resumen después de la limpieza:")
print(f"Clientes con contacto completo: {df_clientes_limpio['tiene_contacto_completo'].sum()}")
print(f"Clientes sin preferencia: {(df_clientes_limpio['preferencia_categoria'] == 'Sin Preferencia').sum()}")
print("\n📌 Primeras filas:")
display(df_clientes_limpio.head())

## Parte 4: Joins y Combinaciones de Datos

### 🔗 Unir múltiples datasets

Vamos a combinar los diferentes datasets para crear vistas unificadas que nos permitan realizar análisis más complejos.

In [0]:
# Unir detalles_ventas con productos para obtener información del producto
df_ventas_detalle = df_detalles_ventas.merge(
    df_productos[['producto_id', 'nombre', 'categoria', 'costo_unitario']],
    on='producto_id',
    how='left'
)

# Unir con ventas para obtener fecha y sucursal
df_ventas_detalle = df_ventas_detalle.merge(
    df_ventas[['venta_id', 'fecha', 'sucursal_id', 'cliente_id']],
    on='venta_id',
    how='left'
)

print("✅ Dataset de ventas completo creado")
print(f"\n📊 Total de registros: {len(df_ventas_detalle):,}")
print(f"\n📌 Columnas disponibles: {list(df_ventas_detalle.columns)}")
print("\n📄 Primeras 5 filas:")
display(df_ventas_detalle.head())

In [0]:
# Calcular costo total y ganancia por línea de venta
df_ventas_detalle['costo_total'] = df_ventas_detalle['cantidad'] * df_ventas_detalle['costo_unitario']
df_ventas_detalle['ganancia'] = df_ventas_detalle['subtotal'] - df_ventas_detalle['costo_total']
df_ventas_detalle['margen_ganancia'] = (df_ventas_detalle['ganancia'] / df_ventas_detalle['subtotal'] * 100).round(2)

print("✅ Métricas de rentabilidad calculadas")
print("\n💰 Resumen de rentabilidad:")
print(f"Ganancia total: ${df_ventas_detalle['ganancia'].sum():,.2f}")
print(f"Margen promedio: {df_ventas_detalle['margen_ganancia'].mean():.2f}%")
print("\n📄 Muestra de datos con métricas:")
display(df_ventas_detalle[['nombre', 'categoria', 'cantidad', 'subtotal', 'costo_total', 'ganancia', 'margen_ganancia']].head(10))

In [0]:
# Agregar nombres de sucursales al dataset
df_ventas_completo = df_ventas.merge(
    df_sucursales[['sucursal_id', 'nombre', 'zona']],
    on='sucursal_id',
    how='left'
)

# Renombrar la columna 'nombre' a 'nombre_sucursal' para evitar confusión
df_ventas_completo = df_ventas_completo.rename(columns={'nombre': 'nombre_sucursal'})

print("✅ Ventas con información de sucursales")
print("\n🏢 Ventas por sucursal:")
resumen_sucursales = df_ventas_completo.groupby('nombre_sucursal').agg({
    'venta_id': 'count',
    'total': 'sum'
}).rename(columns={
    'venta_id': 'cantidad_ventas',
    'total': 'facturacion_total'
}).round(2)

display(resumen_sucursales)

## Parte 5: Integración con SQL

### 📊 Crear vistas temporales para consultas SQL

Databricks permite crear vistas temporales desde DataFrames de Pandas y consultarlas con SQL.

In [0]:
# Crear vistas temporales desde los DataFrames
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Convertir pandas DataFrames a Spark DataFrames y crear vistas
spark.createDataFrame(df_productos).createOrReplaceTempView("productos")
spark.createDataFrame(df_ventas_completo).createOrReplaceTempView("ventas")
spark.createDataFrame(df_ventas_detalle).createOrReplaceTempView("ventas_detalle")
spark.createDataFrame(df_clientes_limpio).createOrReplaceTempView("clientes")

print("✅ Vistas temporales creadas:")
print("  - productos")
print("  - ventas")
print("  - ventas_detalle")
print("  - clientes")
print("\n📊 Ahora puedes usar SQL en las siguientes celdas")

In [0]:
%sql
-- Top 10 productos más vendidos
SELECT 
  nombre,
  categoria,
  SUM(cantidad) as total_vendido,
  SUM(subtotal) as facturacion,
  COUNT(DISTINCT venta_id) as numero_ventas
FROM ventas_detalle
GROUP BY nombre, categoria
ORDER BY total_vendido DESC
LIMIT 10

In [0]:
%sql
-- Ventas por categoría y día de la semana
SELECT 
  v.dia_semana,
  vd.categoria,
  COUNT(DISTINCT vd.venta_id) as cantidad_ventas,
  SUM(vd.subtotal) as facturacion_total,
  AVG(vd.subtotal) as ticket_promedio
FROM ventas v
JOIN ventas_detalle vd ON v.venta_id = vd.venta_id
GROUP BY v.dia_semana, vd.categoria
ORDER BY cantidad_ventas DESC

In [0]:
%sql
-- Análisis de rentabilidad por categoría
SELECT 
  categoria,
  COUNT(*) as lineas_venta,
  SUM(cantidad) as unidades_vendidas,
  SUM(subtotal) as ingresos,
  SUM(costo_total) as costos,
  SUM(ganancia) as ganancia_total,
  ROUND(AVG(margen_ganancia), 2) as margen_promedio
FROM ventas_detalle
GROUP BY categoria
ORDER BY ganancia_total DESC

## Parte 6: Análisis Exploratorio de Datos (EDA)

### 🔍 Identificar patrones y tendencias

Vamos a realizar un análisis exploratorio para descubrir patrones interesantes en los datos.

In [0]:
# Análisis de ventas por mes y año
ventas_mensuales = df_ventas_completo.groupby(['anio', 'mes']).agg({
    'venta_id': 'count',
    'total': 'sum'
}).rename(columns={
    'venta_id': 'cantidad_ventas',
    'total': 'facturacion'
}).reset_index()

ventas_mensuales['facturacion'] = ventas_mensuales['facturacion'].round(2)
ventas_mensuales['ticket_promedio'] = (ventas_mensuales['facturacion'] / ventas_mensuales['cantidad_ventas']).round(2)

print("📈 VENTAS MENSUALES")
print("=" * 80)
display(ventas_mensuales.head(15))

print("\n📊 Estadísticas:")
print(f"Mes con más ventas: {ventas_mensuales.loc[ventas_mensuales['cantidad_ventas'].idxmax(), ['anio', 'mes', 'cantidad_ventas']].values}")
print(f"Mes con mayor facturación: {ventas_mensuales.loc[ventas_mensuales['facturacion'].idxmax(), ['anio', 'mes', 'facturacion']].values}")

In [0]:
# Análisis de patrones por día de semana
ventas_dia_semana = df_ventas_completo.groupby(['dia_semana', 'es_fin_de_semana']).agg({
    'venta_id': 'count',
    'total': ['sum', 'mean']
}).round(2)

ventas_dia_semana.columns = ['cantidad_ventas', 'facturacion_total', 'ticket_promedio']
ventas_dia_semana = ventas_dia_semana.reset_index()

# Ordenar por día de semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
ventas_dia_semana['orden'] = ventas_dia_semana['dia_semana'].apply(lambda x: orden_dias.index(x))
ventas_dia_semana = ventas_dia_semana.sort_values('orden').drop('orden', axis=1)

print("📅 VENTAS POR DÍA DE SEMANA")
print("=" * 80)
display(ventas_dia_semana)

print("\n💡 Insights:")
print(f"Día con más ventas: {ventas_dia_semana.loc[ventas_dia_semana['cantidad_ventas'].idxmax(), 'dia_semana']}")
print(f"Día con menor ventas: {ventas_dia_semana.loc[ventas_dia_semana['cantidad_ventas'].idxmin(), 'dia_semana']}")
print(f"Ticket promedio fin de semana: ${ventas_dia_semana[ventas_dia_semana['es_fin_de_semana'] == True]['ticket_promedio'].mean():.2f}")
print(f"Ticket promedio días hábiles: ${ventas_dia_semana[ventas_dia_semana['es_fin_de_semana'] == False]['ticket_promedio'].mean():.2f}")

In [0]:
# Identificar los productos más rentables
rentabilidad_productos = df_ventas_detalle.groupby(['nombre', 'categoria']).agg({
    'cantidad': 'sum',
    'subtotal': 'sum',
    'ganancia': 'sum',
    'margen_ganancia': 'mean'
}).round(2).reset_index()

rentabilidad_productos = rentabilidad_productos.rename(columns={
    'cantidad': 'unidades_vendidas',
    'subtotal': 'ingresos',
    'ganancia': 'ganancia_total',
    'margen_ganancia': 'margen_promedio'
})

# Top 10 productos por ganancia
top_rentables = rentabilidad_productos.nlargest(10, 'ganancia_total')

print("💰 TOP 10 PRODUCTOS MÁS RENTABLES")
print("=" * 80)
display(top_rentables)

print(f"\n🎯 La ganancia total de estos 10 productos representa: ${top_rentables['ganancia_total'].sum():,.2f}")

## Parte 7: Ejercicios Prácticos

### ✍️ Ejercicios para Resolver

Aplica lo aprendido para resolver los siguientes ejercicios.

#### **Ejercicio 1**: Clientes más frecuentes
Identifica los 10 clientes que más veces han comprado (más transacciones). Muestra su cliente_id, nombre, cantidad de compras y total gastado.

In [0]:
# EJERCICIO 1: Clientes más frecuentes
# Pista: Filtra ventas donde cliente_id no sea nulo, agrupa por cliente_id y une con df_clientes

# Tu código aquí:
clientes_frecuentes = df_ventas_completo[df_ventas_completo['cliente_id'].notna()].groupby('cliente_id').agg({
    'venta_id': 'count',
    'total': 'sum'
}).rename(columns={
    'venta_id': 'numero_compras',
    'total': 'total_gastado'
}).reset_index()

# Unir con información del cliente
clientes_frecuentes = clientes_frecuentes.merge(
    df_clientes_limpio[['cliente_id', 'nombre', 'es_vip']],
    on='cliente_id',
    how='left'
)

top_10_clientes = clientes_frecuentes.nlargest(10, 'numero_compras')

print("🌟 TOP 10 CLIENTES MÁS FRECUENTES")
print("=" * 80)
display(top_10_clientes)

#### **Ejercicio 2**: Estacionalidad de categorías
Analiza cuál categoría de productos se vende más en cada trimestre del año 2024.

In [0]:
# EJERCICIO 2: Estacionalidad de categorías
# Pista: Filtra por año 2024, une ventas_detalle con ventas, agrupa por trimestre y categoría

# Tu código aquí:
ventas_2024 = df_ventas_detalle[df_ventas_detalle['fecha'].dt.year == 2024].copy()
ventas_2024['trimestre'] = ventas_2024['fecha'].dt.quarter

estacionalidad = ventas_2024.groupby(['trimestre', 'categoria']).agg({
    'cantidad': 'sum',
    'subtotal': 'sum'
}).rename(columns={
    'cantidad': 'unidades_vendidas',
    'subtotal': 'facturacion'
}).reset_index()

print("📅 ESTACIONALIDAD POR TRIMESTRE - AÑO 2024")
print("=" * 80)
display(estacionalidad.sort_values(['trimestre', 'facturacion'], ascending=[True, False]))

# Categoría top por trimestre
for trim in [1, 2, 3, 4]:
    top_cat = estacionalidad[estacionalidad['trimestre'] == trim].nlargest(1, 'facturacion')
    print(f"\nQ{trim}: {top_cat['categoria'].values[0]} - ${top_cat['facturacion'].values[0]:,.2f}")

#### **Ejercicio 3**: Consulta SQL personalizada
Escribe una consulta SQL que encuentre cuáles productos tienen un margen de ganancia superior al 55% y cuánto han vendido.

In [0]:
%sql
-- EJERCICIO 3: Productos con alto margen
-- Pista: Filtra por margen_ganancia > 55 y agrupa por producto

SELECT 
  nombre,
  categoria,
  ROUND(AVG(margen_ganancia), 2) as margen_promedio,
  SUM(cantidad) as unidades_vendidas,
  SUM(subtotal) as facturacion,
  SUM(ganancia) as ganancia_total
FROM ventas_detalle
WHERE margen_ganancia > 55
GROUP BY nombre, categoria
ORDER BY ganancia_total DESC

## 🎯 Resumen del TP02

### ✅ Qué aprendimos:

1. **Carga de datos**: Cargamos múltiples archivos CSV con Pandas
2. **Transformaciones**: Creamos columnas derivadas y categorías
3. **Limpieza**: Manejamos valores faltantes y preparamos datos
4. **Joins**: Unimos datasets con `.merge()` para crear vistas unificadas
5. **Integración SQL**: Creamos vistas temporales y ejecutamos consultas SQL
6. **Análisis exploratorio**: Descubrimos patrones temporales y de rentabilidad

### 📊 Insights clave del negocio:

* Los fines de semana tienen mayor volumen de ventas
* Las facturas y el pan son las categorías más rentables
* Existen patrones estacionales en la venta de productos
* Los clientes VIP representan una minoría pero son importantes

### 🚀 Próximos pasos:

En el **TP03** aprenderemos a:
* Crear visualizaciones interactivas con gráficos
* Generar dashboards para presentar insights
* Aplicar perfilado automático de datos
* Comunicar hallazgos de forma efectiva

---

**📝 Excelente trabajo! Ahora tienes las herramientas para manipular y explorar datos en Databricks.**